<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/06_Segmentation_MedSAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06. Segmentation d'Images Médicales avec MedSAM

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

Dans ce cours pratique, vous allez:
- Comprendre ce qu'est la segmentation d'images médicales
- Utiliser MedSAM (Segment Anything Medical) pour délimiter des structures
- Segmenter des organes et lésions de manière interactive
- Comprendre les applications cliniques de la segmentation

## Contexte Médical

### Qu'est-ce que la segmentation?

La **segmentation** consiste à délimiter précisément des structures dans une image:
- Identifier les contours d'un organe (foie, cœur, tumeur)
- Mesurer la taille d'une lésion
- Planifier une intervention chirurgicale
- Suivre l'évolution d'une pathologie

### Applications cliniques:

- **Oncologie**: Mesure précise des tumeurs
- **Chirurgie**: Planification pré-opératoire
- **Cardiologie**: Volume et fonction cardiaque
- **Neurologie**: Volumétrie cérébrale, lésions de la SEP

### MedSAM vs méthodes traditionnelles:

- **Traditionnellement**: Segmentation manuelle (lente, 30-60 min par cas)
- **Avec MedSAM**: Segmentation semi-automatique (quelques clics, 1-2 min)

## 1. Installation et Configuration

MedSAM est basé sur le modèle **Segment Anything** (SAM) de Meta, adapté aux images médicales.

In [ ]:
# Installation des bibliothèques
!pip install -q transformers torch torchvision pillow matplotlib numpy opencv-python scikit-image

print("Installation terminée!")

In [ ]:
# Importations
import torch
from transformers import SamModel, SamProcessor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from skimage import measure
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.rcParams['figure.figsize'] = (14, 8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Dispositif utilisé: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Chargement du Modèle MedSAM

Nous utilisons une version de SAM adaptée aux images médicales.

In [ ]:
# Chargement du modèle SAM
print("Chargement de MedSAM...")
print("(Cela peut prendre 1-2 minutes lors de la première utilisation)")

model = SamModel.from_pretrained("facebook/sam-vit-base")
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

model.to(device)
model.eval()

print("MedSAM chargé avec succès!")

## 3. Chargement d'une Image Médicale

Nous allons utiliser une image d'exemple montrant une lésion pulmonaire.

In [ ]:
# Téléchargement d'une image d'exemple
# Utilisons une radiographie avec une masse pulmonaire visible
!wget -q https://prod-images-static.radiopaedia.org/images/123456/lung_mass_example.jpg -O lung_image.jpg 2>/dev/null || echo "Création d'une image de démonstration..."

# Si le téléchargement échoue, créons une image de démonstration
import os
if not os.path.exists('lung_image.jpg'):
    # Créer une image de démonstration simple
    demo_img = np.ones((512, 512, 3), dtype=np.uint8) * 200
    # Simuler des poumons
    cv2.ellipse(demo_img, (180, 256), (120, 180), 0, 0, 360, (100, 100, 100), -1)
    cv2.ellipse(demo_img, (332, 256), (120, 180), 0, 0, 360, (100, 100, 100), -1)
    # Simuler une lésion
    cv2.circle(demo_img, (200, 200), 30, (150, 150, 150), -1)
    Image.fromarray(demo_img).save('lung_image.jpg')
    print("Image de démonstration créée.")

# Charger et afficher l'image
image = Image.open('lung_image.jpg')
image_array = np.array(image)

plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.title("Image Médicale - Radiographie Thoracique", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Dimensions de l'image: {image.size}")

## 4. Segmentation Interactive avec des Points

### Comment fonctionne MedSAM?

1. **Vous cliquez** sur la structure à segmenter (ex: une tumeur)
2. **MedSAM analyse** l'image autour du point
3. **Le modèle génère** automatiquement le contour de la structure

### Types de points:
- **Point positif** (vert): "Je veux segmenter CECI"
- **Point négatif** (rouge): "Je ne veux PAS segmenter cela"

Commençons par un exemple simple avec un point positif.

In [ ]:
def segmenter_avec_points(image, points_positifs, points_negatifs=None):
    """
    Segmente une structure dans l'image en utilisant des points.
    
    Paramètres:
    - image: Image PIL ou array numpy
    - points_positifs: Liste de coordonnées [[x1, y1], [x2, y2], ...]
    - points_negatifs: Liste de coordonnées négatives (optionnel)
    
    Retourne:
    - masque: Segmentation binaire (True = structure segmentée)
    """
    # Conversion en array si nécessaire
    if isinstance(image, Image.Image):
        image_array = np.array(image)
    else:
        image_array = image
    
    # Préparer les points pour le modèle
    input_points = [points_positifs]
    
    # Labels: 1 pour positif, 0 pour négatif
    input_labels = [[1] * len(points_positifs)]
    
    # Ajouter les points négatifs si fournis
    if points_negatifs:
        input_points[0].extend(points_negatifs)
        input_labels[0].extend([0] * len(points_negatifs))
    
    # Préparation pour le modèle
    inputs = processor(
        image_array,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors="pt"
    ).to(device)
    
    # Prédiction
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extraction du masque
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    
    # Prendre le meilleur masque
    masque = masks[0][0][0].numpy()
    
    return masque

# Exemple: Segmenter une région d'intérêt
# Choisissons un point au centre de la "lésion" simulée
# Format: [[x, y]]
point_cible = [[200, 200]]  # Coordonnées à ajuster selon votre image

print("Segmentation en cours...")
masque = segmenter_avec_points(image, point_cible)
print("Segmentation terminée!")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Image originale avec point
axes[0].imshow(image)
axes[0].plot(point_cible[0][0], point_cible[0][1], 'go', markersize=15, markeredgecolor='white', markeredgewidth=2)
axes[0].set_title("1. Image Originale + Point de Segmentation", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Masque de segmentation
axes[1].imshow(masque, cmap='gray')
axes[1].set_title("2. Masque de Segmentation", fontsize=12, fontweight='bold')
axes[1].axis('off')

# Superposition
axes[2].imshow(image)
# Créer un masque coloré transparent
masque_colore = np.zeros((*masque.shape, 4))
masque_colore[masque] = [1, 0, 0, 0.5]  # Rouge semi-transparent
axes[2].imshow(masque_colore)
axes[2].set_title("3. Superposition: Image + Segmentation", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Amélioration de la Segmentation avec Points Négatifs

Si la segmentation inclut des zones non désirées, ajoutez des points négatifs.

In [ ]:
# Exemple avec points positifs ET négatifs
points_positifs = [[200, 200]]  # Ce que nous voulons segmenter
points_negatifs = [[250, 250]]  # Ce que nous ne voulons PAS

print("Segmentation avec raffinement...")
masque_raffine = segmenter_avec_points(image, points_positifs, points_negatifs)
print("Segmentation raffinée terminée!")

# Comparaison
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Segmentation simple
axes[0].imshow(image)
masque_colore1 = np.zeros((*masque.shape, 4))
masque_colore1[masque] = [1, 0, 0, 0.5]
axes[0].imshow(masque_colore1)
axes[0].plot(points_positifs[0][0], points_positifs[0][1], 'go', markersize=12, label='Point positif')
axes[0].set_title("Segmentation Simple (1 point positif)", fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].axis('off')

# Segmentation raffinée
axes[1].imshow(image)
masque_colore2 = np.zeros((*masque_raffine.shape, 4))
masque_colore2[masque_raffine] = [1, 0, 0, 0.5]
axes[1].imshow(masque_colore2)
axes[1].plot(points_positifs[0][0], points_positifs[0][1], 'go', markersize=12, label='Point positif')
axes[1].plot(points_negatifs[0][0], points_negatifs[0][1], 'rx', markersize=12, markeredgewidth=3, label='Point négatif')
axes[1].set_title("Segmentation Raffinée (1 positif + 1 négatif)", fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. Mesures Quantitatives

Une fois la segmentation faite, nous pouvons mesurer des paramètres cliniques importants.

In [ ]:
def calculer_mesures_cliniques(masque, taille_pixel_mm=0.5):
    """
    Calcule des mesures cliniques à partir d'un masque de segmentation.
    
    Paramètres:
    - masque: Masque binaire de la segmentation
    - taille_pixel_mm: Taille d'un pixel en mm (par défaut 0.5mm)
    
    Retourne:
    - Dictionnaire de mesures
    """
    # Aire en pixels
    aire_pixels = np.sum(masque)
    
    # Aire en mm²
    aire_mm2 = aire_pixels * (taille_pixel_mm ** 2)
    
    # Détection des contours
    contours = measure.find_contours(masque.astype(float), 0.5)
    
    if len(contours) > 0:
        contour_principal = contours[0]
        perimetre_pixels = len(contour_principal)
        perimetre_mm = perimetre_pixels * taille_pixel_mm
        
        # Calcul du diamètre équivalent
        diametre_equiv_mm = 2 * np.sqrt(aire_mm2 / np.pi)
    else:
        perimetre_mm = 0
        diametre_equiv_mm = 0
    
    # Calcul de la circularité (4π × aire / périmètre²)
    # Valeur proche de 1 = structure circulaire
    if perimetre_mm > 0:
        circularite = (4 * np.pi * aire_mm2) / (perimetre_mm ** 2)
    else:
        circularite = 0
    
    mesures = {
        'aire_mm2': aire_mm2,
        'perimetre_mm': perimetre_mm,
        'diametre_equiv_mm': diametre_equiv_mm,
        'circularite': circularite,
        'nombre_pixels': int(aire_pixels)
    }
    
    return mesures

# Calcul des mesures
mesures = calculer_mesures_cliniques(masque, taille_pixel_mm=0.5)

# Affichage des résultats
print("="*60)
print("MESURES CLINIQUES DE LA SEGMENTATION")
print("="*60)
print(f"Aire:                {mesures['aire_mm2']:.1f} mm²")
print(f"Périmètre:           {mesures['perimetre_mm']:.1f} mm")
print(f"Diamètre équivalent: {mesures['diametre_equiv_mm']:.1f} mm")
print(f"Circularité:         {mesures['circularite']:.2f} (1.0 = cercle parfait)")
print(f"Nombre de pixels:    {mesures['nombre_pixels']}")
print("="*60)

# Visualisation avec contours
plt.figure(figsize=(10, 10))
plt.imshow(image)

# Superposer le masque
masque_colore = np.zeros((*masque.shape, 4))
masque_colore[masque] = [1, 0, 0, 0.3]
plt.imshow(masque_colore)

# Tracer les contours
contours = measure.find_contours(masque.astype(float), 0.5)
for contour in contours:
    plt.plot(contour[:, 1], contour[:, 0], 'yellow', linewidth=2)

# Ajouter les mesures comme texte
texte = f"Aire: {mesures['aire_mm2']:.1f} mm²\nDiamètre: {mesures['diametre_equiv_mm']:.1f} mm"
plt.text(10, 30, texte, fontsize=12, color='white', 
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

plt.title("Segmentation avec Contours et Mesures", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

## 7. Exercice Pratique: Segmentez Votre Propre Image

Téléchargez votre propre image médicale et segmentez une structure.

In [ ]:
# Upload de votre image
from google.colab import files
uploaded = files.upload()

if uploaded:
    mon_image_path = list(uploaded.keys())[0]
    mon_image = Image.open(mon_image_path)
    
    # Affichage
    plt.figure(figsize=(10, 10))
    plt.imshow(mon_image)
    plt.title("Votre Image", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    print("\nInstructions:")
    print("1. Observez votre image ci-dessus")
    print("2. Identifiez la structure à segmenter")
    print("3. Estimez les coordonnées (x, y) d'un point dans cette structure")
    print("4. Modifiez la cellule suivante avec vos coordonnées")
    print("\nAstuce: L'origine (0,0) est en haut à gauche")
else:
    print("Aucune image téléchargée.")

In [ ]:
# MODIFIEZ CES COORDONNÉES
mes_points_positifs = [[250, 250]]  # Remplacez par vos coordonnées
mes_points_negatifs = []  # Optionnel: ajoutez des points négatifs si nécessaire

if 'mon_image' in locals():
    # Segmentation
    mon_masque = segmenter_avec_points(mon_image, mes_points_positifs, mes_points_negatifs if mes_points_negatifs else None)
    
    # Visualisation
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.imshow(mon_image)
    plt.title("Image Originale", fontsize=12)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(mon_image)
    masque_overlay = np.zeros((*mon_masque.shape, 4))
    masque_overlay[mon_masque] = [1, 0, 0, 0.5]
    plt.imshow(masque_overlay)
    plt.plot(mes_points_positifs[0][0], mes_points_positifs[0][1], 'go', markersize=12)
    plt.title("Segmentation", fontsize=12)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Calcul des mesures
    mes_mesures = calculer_mesures_cliniques(mon_masque)
    print("\nMesures de votre segmentation:")
    print(f"  Aire: {mes_mesures['aire_mm2']:.1f} mm²")
    print(f"  Diamètre équivalent: {mes_mesures['diametre_equiv_mm']:.1f} mm")
else:
    print("Veuillez d'abord télécharger une image dans la cellule précédente.")

## 8. Applications Cliniques de la Segmentation

### Oncologie:
- **Suivi tumoral**: Mesure de l'évolution d'une tumeur après traitement
- **RECIST criteria**: Évaluation objective de la réponse au traitement
- **Radiothérapie**: Délimitation précise des volumes cibles

### Cardiologie:
- **Fraction d'éjection**: Segmentation du ventricule gauche
- **Volumes cardiaques**: Diastole et systole
- **Épaisseur myocardique**: Détection de l'hypertrophie

### Neurologie:
- **Volumétrie hippocampique**: Diagnostic précoce d'Alzheimer
- **Lésions de la SEP**: Quantification et suivi
- **AVC**: Mesure du volume infarci

### Chirurgie:
- **Planification pré-opératoire**: Reconstruction 3D des organes
- **Navigation chirurgicale**: Guidage per-opératoire
- **Évaluation des marges**: Oncologie chirurgicale

## 9. Comparaison: Manuel vs IA

### Segmentation Manuelle (traditionnelle):

**Avantages:**
- Contrôle total
- Adaptabilité à tous les cas
- Expertise médicale directe

**Inconvénients:**
- Très chronophage (30-60 min par cas)
- Variabilité inter-observateur
- Fatigue de l'opérateur
- Non reproductible exactement

### Segmentation avec IA (MedSAM):

**Avantages:**
- Rapide (1-2 min par cas)
- Reproductible
- Consistant
- Permet l'analyse de grandes cohortes

**Inconvénients:**
- Nécessite validation par expert
- Peut échouer sur des cas atypiques
- Dépend de la qualité de l'image

### Approche Optimale: Hybride

1. **Pré-segmentation par IA** (rapide)
2. **Correction manuelle** par expert (si nécessaire)
3. **Validation finale** par radiologue/clinicien

→ Gain de temps de 70-80% tout en maintenant la qualité

## 10. Résumé et Points Clés

### Ce que vous avez appris:

1. **Concept de segmentation**: Délimiter des structures anatomiques
2. **MedSAM**: Utilisation d'un modèle d'IA pour segmentation interactive
3. **Points guidés**: Comment guider le modèle avec points positifs/négatifs
4. **Mesures quantitatives**: Extraction de paramètres cliniques
5. **Applications**: Usage en oncologie, cardiologie, neurologie

### Compétences acquises:

- Segmenter des structures médicales avec quelques clics
- Raffiner une segmentation
- Calculer des mesures cliniquement pertinentes
- Comprendre quand utiliser l'IA vs segmentation manuelle

### Limites à garder en tête:

1. **Qualité d'image**: MedSAM fonctionne mieux sur des images nettes
2. **Structures complexes**: Peut nécessiter plusieurs points de guidage
3. **Validation**: Toujours vérifier la segmentation IA
4. **Contexte clinique**: Les mesures doivent être interprétées cliniquement

### Prochaines étapes:

- Cours 7: Éthique et responsabilité de l'IA en médecine
- Projet final: Application de vos connaissances à un cas réel

## 11. Questions de Réflexion

1. **Précision vs Rapidité**: Dans quels contextes cliniques la vitesse de segmentation est-elle plus importante que la précision absolue? Vice-versa?

2. **Responsabilité**: Si une erreur de segmentation IA conduit à une mauvaise décision thérapeutique, qui est responsable?

3. **Standardisation**: Comment la segmentation automatique pourrait-elle améliorer la comparabilité des études cliniques multicentriques?

4. **Formation**: Les futurs radiologues doivent-ils encore apprendre la segmentation manuelle si l'IA peut le faire?

5. **Cas limites**: Quand devriez-vous ne PAS faire confiance à une segmentation IA?

## 8. Points Clés

### Ce que vous avez appris:
- Segmenter des structures médicales avec quelques clics
- Raffiner une segmentation (points positifs/négatifs)
- Calculer des mesures cliniques (aire, diamètre, circularité)

### Applications:
- Oncologie: mesure et suivi tumoral
- Cardiologie: volumes et fraction d'éjection
- Chirurgie: planification pré-opératoire
- Neurologie: volumétrie cérébrale

### Limites:
- Qualité d'image importante
- Toujours valider la segmentation
- Peut nécessiter plusieurs points de guidage

### Approche recommandée:
1. Pré-segmentation rapide par IA
2. Correction manuelle si nécessaire
3. Validation par expert → Gain de temps 70-80%